<a href="https://colab.research.google.com/github/adrinorosario/language-model-foundations/blob/main/dissecting_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dissecting LLMs to understand how they work

In this notebook, I dissect multiple models, starting from the smallest LLMs to the largest ones currently out there (as long as colab's runtime can support them).

This was inspired by [Julia Turc's](https://juliaturc.com/$0) youtube video [Discover How LLMs Work by Dissecting Llama](https://www.youtube.com/watch?v=I_2DsaJ4Ung&t=270s$0).

In [1]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab Secrets
token = userdata.get('HF_TOKEN')

if token:
    login(token)
    print("Successfully linked Hugging Face to Colab!")
else:
    print("HF_TOKEN not found in Secrets.")


Successfully linked Hugging Face to Colab!


In [2]:
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [3]:
messages = [
    {
        "role": "user",
        "content": "what is the true meaning of the word 'you'?"
    }
]

outputs = pipe(
    messages,
    max_new_tokens=512
)
outputs

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_ou

[{'generated_text': [{'role': 'user',
    'content': "what is the true meaning of the word 'you'?"},
   {'role': 'assistant',
    'content': 'The word "you" can be a bit ambiguous, as it can refer to different entities in a context-dependent manner. Here are a few possible interpretations:\n\n1. **First-person pronoun**: "You" is a first-person pronoun, which refers to the speaker or the person performing an action. It is used to indicate the subject of a sentence, such as "I," "he," "she," or "they."\n2. **Collective noun**: In some contexts, "you" can refer to a group of people or a collective noun, such as "the crowd," "the team," or "the students."\n3. **Abstract concept**: In philosophy, "you" can also refer to an abstract concept or a personification, representing the human experience or emotions.\n4. **Metaphorical usage**: "You" can be used metaphorically to refer to something or someone else, such as "I\'m doing it for you" (meaning "I\'m doing it for your benefit").\n5. **Abs

In [4]:
messages = [
    { "role": "user", "content": "in less than 50 words, describe what you understand when i ask you 'who are you?'"}
]

outputs = pipe(
    messages,
    max_new_tokens=256
)
outputs

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user',
    'content': "in less than 50 words, describe what you understand when i ask you 'who are you?'"},
   {'role': 'assistant',
    'content': "When you ask me 'who are you', I assume you're asking about my identity, purpose, or function. I'm an AI designed to assist and communicate with users through text-based conversations. I don't have a personal identity or consciousness, but rather a set of pre-programmed responses and knowledge to provide helpful and informative answers."}]}]

An LLM is a neural network. The same as any model, it follows a patter/architecture/blueprint:


$$ x \rightarrow f, w \rightarrow y $$ where:


*   $x$: Inputs that the model receives
*   $f$: The function, i.e., the model architecture. For example, this would be the transformer architecture spelled out in code
*   $w$: Weights the model will learn; also called parameters. They're not hardcoded; set to random that the model will learn to adjust as it learns the data it is fed
*   $y$: Ouput produced by the model

$f, x$ essentially make up the model, to look like this:

$$
x \rightarrow \boxed{f, w} \rightarrow y
$$

In [5]:
pipe.model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

Llama (this model here) is organised as a stack of layers:


*   Embedding layer: (embed_tokens): Embedding(128256, 2048)
*   16 decoder layers: (0-15): 16 x LlamaDecoderLayer
*   Ouptut linear layer: (lm_head): Linear(in_features=2048, out_features=128256, bias=False)

There are multiple other layers inside the broad layer umbrellas.

The bulk processing happens inside the decoding layers, who's fundamental backbone is the [transformer architecture](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf$0)

In [12]:
print(f"Embedding layer of meta-llama/Llama-3.2-1B-Instruct: {embedding_layer}")
print(f"Type of the embedding layer: {type(embedding_layer)}")
print(f"Shape of the embedding layer: {embedding_layer.weight.shape}")

Embedding layer of meta-llama/Llama-3.2-1B-Instruct: Embedding(128256, 2048)
Type of the embedding layer: <class 'torch.nn.modules.sparse.Embedding'>
Shape of the embedding layer: torch.Size([128256, 2048])


In [6]:
type(pipe.model)

transformers.models.llama.modeling_llama.LlamaForCausalLM

**Embedding layer**

1. Tokenize the input. Split the input into smaller tokens, have whitespace markers to preserve original word/sentence boundaries.

2. Encode the tokens into indices. There is a vocabulary underneath $-$ a list of words $-$ using which the current tokens are mapped to based on their positions in that list. This process is deterministic.

3. The indices do not have any special meaning of/on their own. So, we need an extra layer: the embedding layer. This layer converts the indices to vector embeddings in an n-dimensional vector space where similar words that are closely related (e.g. "king" and "queen", or "eiffel" and "tower") are situated close to each other.

$$
\boxed{tokenizer} \rightarrow \boxed{encoder} \rightarrow \boxed{embedder}
$$

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)

embedding_layer = model.get_input_embeddings()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [16]:
sample_text = "Unequivocally"
inputs = tokenizer(sample_text, return_tensors="pt")
input_ids = inputs["input_ids"].to(model.device)

with torch.no_grad():
  embeddings = embedding_layer(input_ids)

print(f"Input Token IDs: {input_ids}")
print(f"Resulting embeddings: {embeddings}")

Input Token IDs: tensor([[128000,  56948,  15780,    511,    750]], device='cuda:0')
Resulting embeddings: tensor([[[ 2.6855e-03,  3.0823e-03, -6.8054e-03,  ...,  1.0757e-03,
           8.2016e-04,  1.5488e-03],
         [ 3.1128e-03,  1.4038e-02, -4.1504e-03,  ...,  1.7700e-02,
           2.2583e-02,  2.0020e-02],
         [-9.1553e-03, -5.2246e-02, -2.8687e-02,  ...,  5.2979e-02,
           2.1973e-03,  1.5869e-02],
         [ 2.5749e-05,  1.9287e-02, -3.1006e-02,  ...,  1.1841e-02,
          -5.8899e-03, -4.4678e-02],
         [ 1.2390e-02, -2.0142e-03, -8.1055e-02,  ..., -8.1177e-03,
          -3.5400e-03,  5.7983e-04]]], device='cuda:0', dtype=torch.bfloat16)


In [18]:
input_ids.shape, embeddings.shape

(torch.Size([1, 5]), torch.Size([1, 5, 2048]))

In [7]:
!pip install torchinfo